In [ ]:
# Install the required libraries
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
faiss-cpu \
sentence-transformers \
openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# Import Libraries
import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI

/tmp/ipykernel_465/3875546470.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
# Mounting Google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# The root directories and pathway
ROOT_DIR = Path("/content/drive/MyDrive/legal_ai/CUAD_v1")
''' TXT_DIR = f"{ROOT_DIR}/full_contract_txt"
this didnt work because there we are treating TXT_DIR as string not file path
check by print(type(TXT_DIR)) '''
TXT_DIR = ROOT_DIR / "full_contract_txt"

In [ ]:
# lets check the connections of txt files
print(type(TXT_DIR))
print(TXT_DIR.exists())

<class 'pathlib.PosixPath'>
True


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "bedrock-api-key-YmVkcm9jay5hbWF6b25hd3MuY29tLz9BY3Rpb249Q2FsbFdpdGhCZWFyZXJUb2tlbiZYLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFTSUFYNjZWV1o0S1pKQzdFNE5NJTJGMjAyNjA2MTklMkZldS1ub3J0aC0xJTJGYmVkcm9jayUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwNjE5VDE2NTM0OVomWC1BbXotRXhwaXJlcz00MzIwMCZYLUFtei1TZWN1cml0eS1Ub2tlbj1JUW9KYjNKcFoybHVYMlZqRVBuJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGd0VhQ21WMUxXNXZjblJvTFRFaVJqQkVBaUJTN0tMYkZkNUFIVUZ3UEd0ZjZWN2hNbCUyRmk0b3ZvMmZ5OVFCenJsUiUyRiUyQlN3SWdPa1E3VUw5S2VheXY1ZiUyQmtrdDdlczJWbk1zeUlHS3hlNWlXbVcxWkVyVEFxcmdNSXd2JTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGQVJBQUdndzFORGMxTVRrMk5EYzFNRGtpRER1MERaM0xHcHFBQmNNVXJ5cUNBN2JUdWh1WDBudkh6OFh6dTM2bkRyOTRRSFZXS0ZWemN2JTJGWVdGJTJGYUt2eU5uSGpIbkNFOUhkMndHTXFuMkZHNmRBWU5ZMm5jdDdSekwxaCUyQnY1dzMyRHc1OFlIdDJhOEM4VUs1VjhEQ1JJY1p5dzlHWHh3UU1NT1ZaQ04yYTJUVE1xWU9QJTJGUGlhY3VaOGtKd1RGV3A1MXdZQlF5cjY2dGhvZTBrQlV1RWRtVHlqQ3RJUEx4eVZ0cVp5eFdIdlAxRHFGS29WcVhGYnNHZ2puMSUyRnJmR2lXJTJGWkdKeEc0Q2F3QU1CZSUyRkhFc2JNOU83ZVR6aiUyRm12WXBZZ0ZSdkNaUDVJMVJ0Q3NmVFpGYUo3Yk1KbFN3JTJCSXpQUGVhM2FHeEhQdG1JaGJaUzZENWtiN1Y5OGhVWVVreDE4NGpHYTFENkNhRTVuMDVPRWxIaUh4RUgxZ3FtYTZYR3FiOXRFb3ElMkJ5VmFQemlrdnl0cFY2NzVVclFLbnI4OVlFaWp3UklYQ3U2SE54VDk3dmt1N1dyelZCS0VPRGhRT2klMkJSWW5FY1g5WnlObyUyRlJvUE85NXZuUmtYdGtkWW9EYVpRNG55JTJCc3h2V1VNcFVxWGdjOEpLMGRuMnNvVUxFQmJCTmJ5ZkVTZXlheWVSNU8yaWkwVCUyQlhncGd4dTFoaTc2SlpHanJ3Rlg5MXdZVDBNTVBqbjFkRUdPdDhDNVoxQUpSRmRxOUo0Q0NJRzB1dkliVmNLNkVtejBxalNpVkJXZTRWeDFocUxaaXVLaiUyQkROcGYlMkZ1YXVHQ292aUt4bkt0Q0NicVY1TVpsQU1tOTAxdmNOdW9CWnlaN05rbnh2OFRNRnNBMjh1ZlBvR2glMkZaVFBTenFLNm9uSm9zanNQblhKUTE2WVhiU0ZPOHRFZXBCdnc4ZWRPNVZpa0tKN3A2Z212RmFsRFRQeW9kSTh0cDVwM29IRXAwVWIyaExrVElLaUZxVSUyQlh1VjVONm9sTk9obnM1NUtWQzYxYUhXNDViUE5mWWhIS0dlZTc2T0xPcE5IRVNCR2wlMkJ5ZlUzSE02Y0duVU5hNWs0c3FpMVQ1STQlMkZiTUgzJTJGbzFJdlN5Ym9DRlF6ZnI5aG1oJTJGeCUyQktPTkczT055emFkdU1HJTJCVms0R1Y0bk0lMkZoaEF0aFFkTWhtbUklMkZZODh5QVJsMFEyRVRsTmN5M25CRmhHdXpVeDd0N2dkSVFEVzZZcjB6aFVxcG44Zkl5akxRb1J6aHU2SVk5YiUyQlpac3FGUFRCR0JIOUZjZXZZZ0g3T0dyaHhDdW1RcFhmbVI2VVlNJTJCUUxkNlhrYUlNMUhtbUpQRU1JZEtQVzdqbFNFNiZYLUFtei1TaWduYXR1cmU9ZDAyY2MzYjk3MTk5MWE1OWRmZTMyNWExNGIzNzRjMTE0Mjc4NWYzNTY0MzliNmVkMDgyNTczNTU0ODkyMDBmZiZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmVmVyc2lvbj0x"
os.environ["OPENAI_BASE_URL"] = "https://bedrock-mantle.eu-north-1.api.aws/v1"

In [ ]:
from openai import OpenAI
client = OpenAI()
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input="What is Retrieval Augmented Generation?"
)

print(response.output_text)

**Retrieval‑Augmented Generation (RAG)**  
*The hybrid approach that couples a “search engine” with a generative language model so that the model can ground its output in up‑to‑date, factual, and domain‑specific information.*

---

## 1. Core Idea in One Sentence
RAG lets a large language model (LLM) **first retrieve relevant documents (or chunks of text) from an external knowledge base** and then **condition its generation on those retrieved passages**, producing answers that are both fluent and fact‑checked against real data.

---

## 2. Why RAG Exists  

| Problem with vanilla LLMs | How RAG Helps |
|---------------------------|----------------|
| **Knowledge cutoff** – the model only knows what was in its training data (e.g., up to 2021 for many public models). | Retrieves *current* information from the web, databases, or private corpora. |
| **Hallucinations** – LLMs can fabricate plausible‑sounding facts. | The generator is tethered to concrete evidence, reducing fabrications. |


Load all contracts

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(TXT_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True
)
documents = loader.load()

100%|██████████| 510/510 [00:10<00:00, 49.91it/s] 


In [ ]:
print("Total Documents:", len(documents))

Total Documents: 510


In [ ]:
print(documents[0].page_content[:1000])
print(documents[0].metadata)

Confidential treatment has been requested with respect to portions of this agreement as indicated by "[***]" and such confidential portions have been deleted and filed separately with the Securities and Exchange Commission pursuant to Rule 24b-2 of the Securities Exchange Act of 1934, as amended. Amendment #3 to the Manufacturing Agreement This Amendment #3 to the Manufacturing Agreement (this "Amendment #3") is made effective as of December 21, 2017 ("Amendment Effective Date"), by and between ADMA BioManufacturing, LLC, a Delaware limited liability company, having a place of business at 5800 Park of Commerce Boulevard NW, Boca Raton, Florida 33487 USA ("ADMA") and Sanofi Pasteur S.A., a company existing and organized under the laws of France ("Sanofi Pasteur"), having its registered head office at 14, espace Henry Vallee, 69007, Lyon, France. WHEREAS, ADMA (as successor-in-interest to Biotest Pharmaceuticals Corporation ("BPC") and Sanofi Pasteur are parties to that certain Manufactu

In [ ]:
print(type(documents))
print(type(documents[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [ ]:
print(documents[0].metadata.keys())

dict_keys(['source'])


In [ ]:
'''
This will be useful later for:
 Filter only Manufacturing contracts,
 Filter only Strategic Alliance contracts, and
 Contract analytics
'''

for doc in documents:
    filename = Path(doc.metadata["source"]).name
    doc.metadata = {
        "source": filename
    }

In [ ]:
print(documents[85].metadata)

{'source': 'ClickstreamCorp_20200330_1-A_EX1A-6 MAT CTRCT_12089935_EX1A-6 MAT CTRCT_Development Agreement.txt'}


Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)


In [ ]:
print("Total Chunks:", len(chunks))
print(type(chunks))
print(type(chunks[0]))

Total Chunks: 39025
<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [ ]:
chunks[45].page_content

'(v) The MHCs and the Company have filed with the Office of Thrift Supervision (the "OTS") an application for approval of their acquisition of the Bank (the "Holding Company Application") on Form H-(e)1 promulgated under the savings and loan holding company provisions of the Home Owners\' Loan Act, as amended ("HOLA") and the regulations promulgated thereunder. The Holding Company Application includes a proxy statement for the special meeting of stockholders of the Bank called to approve the Agreement and Plan of Reorganization (the "Proxy Statement"). The MHCs and the Company have received written notice from the OTS of its approval of the Holding Company Application, such approval remains in full force and effect and no order has been issued by the OTS suspending or revoking such approval and no proceedings therefor have been initiated or threatened by the OTS. At the date of such approval and at the Closing Time referred to in Section 2, the Holding Company Application complied and'

In [ ]:
chunks[45].metadata

{'source': 'ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt'}

In [ ]:
# Chunk statistics
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Min Chunk Length :", min(chunk_lengths))
print("Max Chunk Length :", max(chunk_lengths))
print("Average Length   :", sum(chunk_lengths)/len(chunk_lengths))

Min Chunk Length : 1
Max Chunk Length : 1000
Average Length   : 762.9654836643177


In [ ]:
small_chunks = [
    chunk
    for chunk in chunks
    if len(chunk.page_content) < 20
]

print("Tiny Chunks:", len(small_chunks))

Tiny Chunks: 529


In [ ]:
for chunk in small_chunks[:]:
    print("-"*50)
    print(repr(chunk.page_content))

--------------------------------------------------
'-5-'
--------------------------------------------------
'-15-'
--------------------------------------------------
'-18-'
--------------------------------------------------
'-23-'
--------------------------------------------------
'-24-'
--------------------------------------------------
'becomes aware of.'
--------------------------------------------------
'th'
--------------------------------------------------
'th'
--------------------------------------------------
'8'
--------------------------------------------------
'28'
--------------------------------------------------
'3'
--------------------------------------------------
'3.16 Information'
--------------------------------------------------
'2'
--------------------------------------------------
'3'
--------------------------------------------------
'4'
--------------------------------------------------
'5'
--------------------------------------------------
'6'
-----------------

In [ ]:
#  lets remove the noise by removing the smaller chunks
print("Chunks Before Cleaning:", len(chunks))

Chunks Before Cleaning: 39025


In [ ]:
clean_chunks = []

for chunk in chunks:
    text = chunk.page_content.strip()
    if len(text) < 50:
        continue
    clean_chunks.append(chunk)

chunks = clean_chunks

In [ ]:
print("Chunks After Cleaning:", len(chunks))

Chunks After Cleaning: 37818


In [ ]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Min Chunk Length :", min(chunk_lengths))
print("Max Chunk Length :", max(chunk_lengths))
print("Average Length   :", sum(chunk_lengths)/len(chunk_lengths))

Min Chunk Length : 50
Max Chunk Length : 1000
Average Length   : 786.5126923687133


In [ ]:
print("="*60)
print("LEGAL RAG PREPROCESSING SUMMARY")
print("="*60)

print("Documents Loaded :", len(documents))
print("Chunks Created   :", len(chunks))

print(
    "Chunks/Contract  :",
    round(len(chunks)/len(documents), 2)
)

print(
    "Average Chunk Size:",
    round(sum(chunk_lengths)/len(chunk_lengths),2)
)

LEGAL RAG PREPROCESSING SUMMARY
Documents Loaded : 510
Chunks Created   : 37818
Chunks/Contract  : 74.15
Average Chunk Size: 786.51


Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# Embedding Dimension : 384
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

/tmp/ipykernel_465/2841826385.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sample_embedding = embeddings.embed_query(
    "What is Rag?"
)

print(type(sample_embedding))
print(len(sample_embedding))

<class 'list'>
384


VectorStore and Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print("FAISS Index Created Successfully")

FAISS Index Created Successfully


In [ ]:
#  Lets save our vectorStore FAISS
FAISS_PATH = "/content/drive/MyDrive/legal_ai/vectorstores/legal_faiss"
vectorstore.save_local(FAISS_PATH)

print("FAISS Saved")


FAISS Saved


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5
    }
)

In [ ]:
query = "What is the governing law?"

results = retriever.invoke(query)

In [ ]:
print(results)

[Document(id='cd1fd674-6679-4feb-831d-d1d908b87774', metadata={'source': 'ChinaRealEstateInformationCorp_20090929_F-1_EX-10.32_4771615_EX-10.32_Content License Agreement.txt'}, page_content="10.11. Governing Law. This Agreement and any dispute or claim arising out of or in connection with it or its subject matter shall be governed by,  and construed in accordance with, the laws of the People's Republic of China (without regard to its conflicts of laws rules that would mandate the  application of the laws of another jurisdiction)."), Document(id='dfb6ecb1-792a-415a-a3cb-c67ad2f6c5d9', metadata={'source': 'OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.txt'}, page_content='14\n\n\n\n\n\nARTICLE VIII\n\nGOVERNING LAW AND DISPUTE RESOLUTION'), Document(id='a22085c0-a5a2-404e-8a51-86788dea7ae9', metadata={'source': 'AtnInternationalInc_20191108_10-Q_EX-10.1_11878541_EX-10.1_Maintena

In [ ]:
for i, doc in enumerate(results):

    print("="*80)

    print(f"RESULT {i+1}")

    print("\nSOURCE:")
    print(doc.metadata["source"])

    print("\nTEXT:")
    print(doc.page_content[:1000])

RESULT 1

SOURCE:
ChinaRealEstateInformationCorp_20090929_F-1_EX-10.32_4771615_EX-10.32_Content License Agreement.txt

TEXT:
10.11. Governing Law. This Agreement and any dispute or claim arising out of or in connection with it or its subject matter shall be governed by,  and construed in accordance with, the laws of the People's Republic of China (without regard to its conflicts of laws rules that would mandate the  application of the laws of another jurisdiction).
RESULT 2

SOURCE:
OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.txt

TEXT:
14





ARTICLE VIII

GOVERNING LAW AND DISPUTE RESOLUTION
RESULT 3

SOURCE:
AtnInternationalInc_20191108_10-Q_EX-10.1_11878541_EX-10.1_Maintenance Agreement.txt

TEXT:
3.14 Governing Law. The laws of the State of New York (excluding any laws that direct the application of another jurisdiction's law) govern all matters arising out of or relat

Generation and building context

In [ ]:
query = "What is the governing law?"

retrieved_docs = retriever.invoke(query)

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

In [ ]:
print(context[:3000])

10.11. Governing Law. This Agreement and any dispute or claim arising out of or in connection with it or its subject matter shall be governed by,  and construed in accordance with, the laws of the People's Republic of China (without regard to its conflicts of laws rules that would mandate the  application of the laws of another jurisdiction).

14





ARTICLE VIII

GOVERNING LAW AND DISPUTE RESOLUTION

3.14 Governing Law. The laws of the State of New York (excluding any laws that direct the application of another jurisdiction's law) govern all matters arising out of or relating to this Agreement and all of the transactions it contemplates, including its validity, interpretation, construction, performance, and enforcement.

3.15 Indemnity

18. GOVERNING LAWS

18.1 Unless otherwise agreed to in writing by the parties, the Agreement shall be governed by and construed in accordance with the laws of the Province of British Columbia and the federal laws of Canada applicable therein, and the 

Legal Prompt

In [ ]:
prompt = f"""
You are a legal contract assistant.

Answer ONLY from the provided context.

If the answer is not found in the context, say:

'Information not found in retrieved contracts.'

Provide a concise legal answer.

QUESTION:
{query}

CONTEXT:
{context}
"""

In [ ]:
# Generate Answer
response = client.responses.create(
    model="openai.gpt-oss-120b",
    input=prompt
)

answer = response.output_text

print(answer)

The agreement contains several governing‑law provisions:

* **Clause 10.11:** The Agreement is governed by the laws of the **People’s Republic of China** (without regard to conflict‑of‑law rules).  
* **Article VIII 3.14 (and 24.1):** Governing law is the **State of New York** (subject to New York’s General Obligations Law).  
* **Section 18.1:** Unless otherwise agreed in writing, the Agreement is governed by the laws of the **Province of British Columbia** and the applicable **federal laws of Canada**, with jurisdiction in the courts of British Columbia.


In [ ]:
# Show citation
print("\n\nSOURCES")

for doc in retrieved_docs:
    print("-", doc.metadata["source"])




SOURCES
- ChinaRealEstateInformationCorp_20090929_F-1_EX-10.32_4771615_EX-10.32_Content License Agreement.txt
- OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.txt
- AtnInternationalInc_20191108_10-Q_EX-10.1_11878541_EX-10.1_Maintenance Agreement.txt
- CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.txt
- AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1.txt


In [ ]:
def ask_legal_rag(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    prompt = f"""
    You are a legal contract assistant.

    Answer only from context.

    Question:
    {question}

    Context:
    {context}
    """

    response = client.responses.create(
        model="openai.gpt-oss-120b",
        input=prompt
    )

    print(response.output_text)

    print("\nSOURCES")

    for doc in retrieved_docs:
        print("-", doc.metadata["source"])

In [ ]:
# Retrieval Quality Test
test_questions = [

    # Governing Law
    "What is the governing law?",

    # Termination
    "What are the termination rights under the agreement?",

    # Confidentiality
    "What confidentiality obligations exist?",

    # Indemnification
    "What indemnification obligations exist?",

    # Liability
    "What limitations of liability exist?",

    # Assignment
    "Can either party transfer its obligations to another company?",

    # Renewal
    "Under what conditions can the agreement be renewed?",

    # Term
    "How long does the agreement remain in effect?",

    # Notice
    "What notice requirements are specified in the agreement?",

    # Payment
    "What payment obligations are imposed on the parties?"
]

In [ ]:
for question in test_questions:

    print("\n")
    print("="*100)
    print("QUESTION:")
    print(question)

    ask_legal_rag(question)

    print("="*100)



QUESTION:
What is the governing law?
**Governing law provisions found in the excerpts**

| Section | Governing law stated |
|---------|----------------------|
| **10.11** | The laws of the **People’s Republic of China** (without regard to its conflict‑of‑law rules). |
| **3.14** (Article VIII) | The laws of the **State of New York** (excluding any statutes that would mandate application of another jurisdiction’s law). |
| **18.1** | The laws of the **Province of British Columbia** and the applicable **federal laws of Canada**, with jurisdiction in the courts of British Columbia. |
| **24.1** | Governing law is **New York law** as set out in Section 5‑1401 of the New York General Obligations Law. |

So, depending on which part of the agreement is being referred to, the governing law could be **China**, **New York (USA)**, or **British Columbia (Canada)**.

SOURCES
- ChinaRealEstateInformationCorp_20090929_F-1_EX-10.32_4771615_EX-10.32_Content License Agreement.txt
- OTISWORLDWIDECORP_

In [ ]:
# Hallucination Test
hallucination_tests = [

    "Who is the CEO of the company?",

    "What is the company's stock price?",

    "What is the weather in New York?",

    "What is the favorite food of the contracting parties?",

    "Which team won the FIFA World Cup?"
]


In [ ]:
for question in hallucination_tests:

    print("\n")
    print("="*100)
    print("QUESTION:")
    print(question)

    ask_legal_rag(question)

    print("="*100)



QUESTION:
Who is the CEO of the company?
The CEO is **Anders Gustafsson**.

SOURCES
- ZEBRATECHNOLOGIESCORP_04_16_2014-EX-10.1-INTELLECTUAL PROPERTY AGREEMENT.txt
- LifewayFoodsInc_20160316_10-K_EX-10.24_9489766_EX-10.24_Endorsement Agreement.txt
- URSCORPNEW_03_17_2014-EX-99-COOPERATION AGREEMENT.txt
- KALLOINC_11_03_2011-EX-10.1-STRATEGIC ALLIANCE AGREEMENT.txt
- VALENCETECHNOLOGYINC_02_14_2003-EX-10-JOINT VENTURE CONTRACT.txt


QUESTION:
What is the company's stock price?
The stock is priced at **$10.00 per share**.

SOURCES
- ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt
- ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement.txt
- AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt
- AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1.txt
- PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.txt


QUESTION:
What is the weather in New York?
The provi

In [ ]:
# Legal Reasoning Tests
reasoning_tests = [

    "Summarize the key risks for a party entering this agreement.",

    "What obligations survive termination?",

    "What protections exist for intellectual property?",

    "Compare confidentiality obligations across agreements.",

    "Identify clauses that may require legal review.",

    "Which provisions create the greatest legal exposure?",

    "What contractual protections are provided to each party?"
]

In [ ]:
for question in reasoning_tests:

    print("\n")
    print("="*100)
    print("QUESTION:")
    print(question)

    ask_legal_rag(question)

    print("="*100)



QUESTION:
Summarize the key risks for a party entering this agreement.
**Key Risks for a Party Joining This Agreement (based on the excerpt provided)**  

| Area | What the clause says | Why it is a risk for the party |
|------|----------------------|--------------------------------|
| **Travel & Participation Costs** | *“Each Party shall be responsible for all travel and related costs … to attend meetings of, and otherwise participate on, a committee.”* | The party must fund all its own staff’s travel, lodging, customs duties, taxes (except income tax), insurance or a risk‑allowance, and any other expenses listed in the schedule. Unexpected trips or prolonged meetings can create significant, unbudgeted out‑of‑pocket costs. |
| **Additional Fees & Administrative Charges** | Fees to the NA (Annex 3), CBP (Annex 4), and custodial costs for distributing certified copies are payable up to the RFS Date. | The party may incur extra, “hard‑cost” payments that are not linked to a specific de

In [ ]:
# Stress Test
stress_tests = [

    "List all governing law jurisdictions mentioned in the retrieved contracts.",

    "Compare termination provisions across the retrieved contracts.",

    "Compare indemnification obligations across agreements.",

    "Compare confidentiality obligations across agreements.",

    "Compare liability limitations across agreements."
]

In [ ]:
for question in stress_tests:

    print("\n")
    print("="*100)
    print("QUESTION:")
    print(question)

    ask_legal_rag(question)

    print("="*100)



QUESTION:
List all governing law jurisdictions mentioned in the retrieved contracts.
- State of Illinois (the internal laws of the State of Illinois)  
- Federal court in Chicago, Illinois  
- State court in DuPage County, Illinois

SOURCES
- EhaveInc_20190515_20-F_EX-4.44_11678816_EX-4.44_License Agreement_ Reseller Agreement.txt
- FIDELITYNATIONALINFORMATIONSERVICES,INC_08_05_2009-EX-10.3-INTELLECTUAL PROPERTY AGREEMENT.txt
- NANOPHASETECHNOLOGIESCORP_11_01_2005-EX-99.1-DISTRIBUTOR AGREEMENT.txt
- BIOFRONTERAAG_04_29_2019-EX-4.17-SUPPLY AGREEMENT.txt
- OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.txt


QUESTION:
Compare termination provisions across the retrieved contracts.
**Comparison of Termination Provisions in the Retrieved Contracts**

| Contract | Who May Terminate | Notice / Cure Requirements | Allowed Reasons / Grounds | Scope of Termination | Post‑Termination Ob

In [ ]:
ask_legal_rag("What confidentiality obligations exist?")

ask_legal_rag("What indemnification obligations exist?")

ask_legal_rag("Summarize the key risks for a party entering this agreement.")

ask_legal_rag("Who is the CEO of the company?")

The parties are bound by several layered confidentiality duties:

1. **General Duty to Protect Confidential Information**  
   - Each party must **prevent unauthorized access** to the other party’s confidential materials.  
   - The party must **instruct its employees, agents, and contractors** (i) of the confidentiality obligations set out in the agreement and (ii) that they may not try to bypass any security procedures or devices.  
   - The party may satisfy this instruction requirement by having those persons sign its **standard‑form confidentiality agreement**, provided that the agreement **reasonably accomplishes the same purposes**.  

2. **Limited Distribution**  
   - Confidential information may be shared **only with persons who have a “need to know”** in order to perform duties related to the agreement.  

3. **Permitted Disclosure Under Legal Compulsion**  
   - A party may disclose the other party’s confidential information if it is **required by subpoena, court order, reg